# Mechanizm pobierania danych z GUS (API DBW)

https://api.stat.gov.pl/Home/DBWApi

https://api-dbw.stat.gov.pl/apidocs/index.html

Pomyślane jako narzędzie do pobrania danych o inflacji (CPI Usługi lekarskie), ale w Main można przejrzeć dane i ew. zmodyfikować filtrowania po nazwie.

## Importy

In [1]:
from requests.exceptions import HTTPError
import pandas as pd
from datetime import datetime
from pathlib import Path
from tqdm import tqdm

from functools import lru_cache
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option("display.max_columns", None)

## Funkcje pomocnicze

In [2]:
def _get_session() -> requests.Session:
    session = requests.Session()
    session.headers.update({"accept": "application/json"})
    retry = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 503])
    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session

### _get_periods

In [3]:
@lru_cache(maxsize=1)
def _get_periods() -> dict:
    """Pobiera słownik okresów statystycznych z API GUS DBW."""
    url = "https://api-dbw.stat.gov.pl/api/dictionaries/periods-dictionary"
    params = {"page-size": 500, "page": 0, "lang": "pl"}

    first = SESSION.get(url, params=params, timeout=10)
    first.raise_for_status()
    first_json = first.json()

    pages = first_json["page-count"]
    data = first_json["data"]

    for page in range(1, pages + 1):
        params["page"] = page
        response = SESSION.get(url, params=params, timeout=10)
        response.raise_for_status()
        data.extend(response.json()["data"])

    return {d["id-okres"]: d["opis"] for d in data}

### get_variables_periods_sections

In [12]:
def get_variables_periods_sections() -> pd.DataFrame:
    """Pobiera pełny zakres zmiennych, przekrojów oraz okresów z API GUS DBW."""
    url = "https://api-dbw.stat.gov.pl/api/variable/variable-section-periods"
    params = {"ile-na-stronie": 1000, "numer-strony": 0, "lang": "pl"}

    first = SESSION.get(url, params=params, timeout=10)
    first.raise_for_status()
    first_json = first.json()

    pages = first_json["page-count"]
    data = first_json["data"]

    for page in range(1, pages + 1):
        params["numer-strony"] = page
        response = SESSION.get(url, params=params, timeout=10)
        response.raise_for_status()
        data.extend(response.json()["data"])

    lookup = _get_periods()
    df = pd.DataFrame(data)
    df["opis-okres"] = df["id-okres"].map(lookup)

    return df

### _get_positions_lookup

In [5]:
@lru_cache(maxsize=128)
def _get_positions_lookup(section_id: int) -> dict:
    """Pobiera słownik pozycji statystycznych dla danego przekroju z API GUS DBW."""
    url = "https://api-dbw.stat.gov.pl/api/variable/variable-section-position"
    params = {"id-przekroj": section_id, "lang": "pl"}

    response = SESSION.get(url, params=params, timeout=10)
    response.raise_for_status()

    return {d["id-pozycja"]: d["nazwa-pozycja"] for d in response.json()}

### _get_ways_of_presentation

In [6]:
@lru_cache(maxsize=1)
def _get_ways_of_presentation() -> dict:
    """Pobiera słownik sposobów prezentacji z API GUS DBW."""
    url = "https://api-dbw.stat.gov.pl/api/dictionaries/way-of-presentation"
    params = {"page": 0, "page-size": 500, "lang": "pl"}

    first = SESSION.get(url, params=params, timeout=10)
    first.raise_for_status()
    first_json = first.json()

    pages = first_json["page-count"]
    data = first_json["data"]

    for page in range(1, pages + 1):
        params["page"] = page
        response = SESSION.get(url, params=params, timeout=10)
        response.raise_for_status()
        data.extend(response.json()["data"])

    return {d["id-sposob-prezentacji-miara"]: d["nazwa"] for d in data}

### get_positions 

In [7]:
def get_positions(section_ids: list[int]) -> pd.DataFrame:
    """
    Pobiera pozycje statystyczne wraz z informacją o wymiarach
    dla wskazanych przekrojów z API GUS DBW.

    Parameters
    ----------
    section_ids : list[int]
        Identyfikatory przekrojów, dla których mają zostać pobrane pozycje.
    """
    url = "https://api-dbw.stat.gov.pl/api/variable/variable-section-position"
    
    dfs = []
    for section_id in section_ids:
        params = {"id-przekroj": section_id, "lang": "pl"}
        response = SESSION.get(url, params=params, timeout=10)
        response.raise_for_status()
        dfs.append(pd.DataFrame(response.json()))

    return pd.concat(dfs, ignore_index=True)

### get_years_to_fetch

In [8]:
def get_years_to_fetch(csv_path: str, variable_id: int, all_years: list[int]) -> list[int]:
    """Zwraca lata których jeszcze nie ma w CSV dla danej zmiennej."""
    if not Path(csv_path).exists():
        return all_years
    
    df_existing = pd.read_csv(csv_path)
    
    if "id-zmienna" not in df_existing.columns or "id-rok" not in df_existing.columns:
        return all_years
    
    fetched_years = df_existing[df_existing["id-zmienna"] == variable_id]["id-rok"].unique()
    
    # Zawsze dociągaj ostatni rok — GUS może go korygować
    last_fetched = max(fetched_years) if len(fetched_years) > 0 else None
    
    return [y for y in all_years if y not in fetched_years or y == last_fetched]

### get_data

In [9]:
def get_data(
    variable_id: int,
    section_ids: list[int],
    year_ids: list[int],
    period_ids: list[int],
    # position_ids: list[int],
    section_names: dict
) -> pd.DataFrame:
    url = "https://api-dbw.stat.gov.pl/api/variable/variable-data-section"
    results = []

    for section_id in tqdm(section_ids):
        for year_id in tqdm(year_ids):
            for period_id in tqdm(period_ids):
                params = {
                    "id-zmienna": variable_id,
                    "id-przekroj": section_id,
                    "id-rok": year_id,
                    "id-okres": period_id,
                    "ile-na-stronie": 500,
                    "numer-strony": 0,
                    "lang": "pl"
                }

                try:
                    first = SESSION.get(url, params=params, timeout=10)
                    print(f"response time: {datetime.now()}, response params: {params}")
                    first.raise_for_status()
                except HTTPError as e:
                    if e.response.status_code == 404:
                        continue
                    raise

                first_json = first.json()
                pages = first_json["page-count"]
                data = first_json["data"]

                for page in tqdm(range(1, pages + 1)):
                    params["numer-strony"] = page
                    response = SESSION.get(url, params=params, timeout=10)
                    print(f"response time: {datetime.now()}, response params: {params}")
                    response.raise_for_status()
                    data.extend(response.json()["data"])

                section_df = pd.DataFrame(data)
                section_df["id-rok"] = year_id
                results.append(section_df)

    if not results:
        return pd.DataFrame()

    df = pd.concat(results, ignore_index=True)
    # df = df[df["id-pozycja-2"].isin(position_ids)]
    print(f"time: {datetime.now()}, All done")

    
    df["opis-okres"] = df["id-okres"].map(_get_periods())
    df["sposob-prezentacji"] = df["id-sposob-prezentacji-miara"].map(_get_ways_of_presentation())

    # Mapowanie pozycji per przekrój
    df["opis-pozycja-2"] = df.apply(
        lambda row: _get_positions_lookup(row["id-przekroj"]).get(row["id-pozycja-2"]), axis=1
    )
    df["opis-pozycja-3"] = df.apply(
        lambda row: _get_positions_lookup(row["id-przekroj"]).get(row["id-pozycja-3"]), axis=1
    )


    df["nazwa-przekroj"] = df["id-przekroj"].map(section_names)

    return df[["nazwa-przekroj", "opis-pozycja-3", "opis-pozycja-2", "opis-okres", "sposob-prezentacji", "id-rok", "wartosc"]]

## Main

### Start sesji

In [10]:
SESSION = _get_session()

### Wybór przekroju i zmiennej

In [11]:
%%time
df = get_variables_periods_sections()

CPU times: total: 5.03 s
Wall time: 1min 50s


In [14]:
variable_filter = "Wskaźniki cen towarów i usług konsumpcyjnych"
variable_id = df[df["nazwa-zmienna"]==variable_filter]["id-zmienna"].unique()[0]

In [15]:
section_ids = list(df[df["id-zmienna"]==variable_id]["id-przekroj"].unique())
section_names = dict(zip(
    df[df["id-zmienna"]==variable_id]["id-przekroj"],
    df[df["id-zmienna"]==variable_id]["nazwa-przekroj"]
))

### Wybór okresu dla przekroju i zmiennej

In [16]:
period_ids = (
    df[
        (df["id-zmienna"] == variable_id) &
        (df["id-przekroj"].isin(section_ids))
    ]["id-okres"]
    .unique().tolist()
)

### Wybór pozycji

In [17]:
df_positions = get_positions(section_ids)

In [18]:
position_ids = df_positions["id-pozycja"].tolist()

In [22]:
import itertools
combinations = list(itertools.product(section_ids, years_to_fetch, period_ids))

### Pobierz wskaźniki

In [20]:
years = list(range(2015, 2025))
years_to_fetch = get_years_to_fetch("data/gus_data.csv", variable_id, years)

In [21]:
variable_id = int(variable_id)
section_ids = [int(x) for x in section_ids]
period_ids = [int(x) for x in period_ids]
position_ids = [int(x) for x in position_ids]

In [ ]:
%%time
df = get_data(variable_id=variable_id,
              section_ids=section_ids, 
              year_ids=years, 
              period_ids=period_ids,
              # position_ids=position_ids,
              section_names=section_names)

In [ ]:
KEY_COLUMNS = ["opis-pozycja-3", "opis-pozycja-2", "opis-okres", "sposob-prezentacji", "nazwa-przekroj", "id-rok"]

if Path("data/gus_data.csv").exists():
    df_existing = pd.read_csv("data/gus_data.csv")
    df_combined = pd.concat([df_existing, df], ignore_index=True)
    df_combined = df_combined.drop_duplicates(subset=KEY_COLUMNS, keep="last")
else:
    df_combined = df

df_combined.to_csv("data/gus_data.csv", index=False)